# E6 — Concentration estimation with known composition

Estimate the nominal concentration of each present component when its presence indicator is provided to the regressor. The target is log10(ppb); this oracle-composition task does not measure deployable mixture identification.

In [ ]:
from pathlib import Path
from io import StringIO
import os, re, sys, json, hashlib, random, copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # saved publication figures; independent of notebook GUI backend
import matplotlib.pyplot as plt
import joblib
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score

ROOT = Path.cwd().resolve()
SRC = next((q for p in (ROOT, *ROOT.parents) for q in (p, p/'src') if (q/'esdata').is_dir()), None)
if SRC is None: raise FileNotFoundError('Run inside ml_flavored-ml (repository root or src/)')
OUT = SRC / 'outputs' / 'E6_concentration_v1'; OUT.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path(os.getenv('ESMELL_DATA_DIR', str(SRC.parent / 'data' / 'raw')))
GASES = ('acetone','isopropanol','pentane')
CHANNELS = ('R1','R2','R11','R12','R16','R17')
SEEDS = (42,43,44)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_EPOCHS, PATIENCE, BATCH = 100, 16, 64
ARCHIVE_STEMS = {130:'curiosity-2v15.130.0',146:'curiosity-2v15.146.0'}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4<<20), b''): h.update(block)
    return h.hexdigest()

def find_archive(card):
    matches=sorted(DATA_DIR.glob(ARCHIVE_STEMS[card]+'.tar*.gz'))
    if len(matches)!=1: raise FileNotFoundError(f'Expected one archive {ARCHIVE_STEMS[card]}.tar*.gz in {DATA_DIR}; found {matches}')
    return matches[0]

assert DATA_DIR.is_dir(), 'Set ESMELL_DATA_DIR to the folder containing card 130/146 tar.gz archives'
ARCHIVES={c:find_archive(c) for c in ARCHIVE_STEMS}
print('device:', DEVICE, '| archives:', {k:v.name for k,v in ARCHIVES.items()}, '| output:', OUT)

## 1. Shared preprocessing and split

Reuse E5-compatible signal extraction and the fixed 130 → 146 chronological cohorts.

In [ ]:
# E5's exact per-sample pipeline signature. The source-level hash prevents stale cached signals.
code_hashes={str(p.relative_to(SRC)):sha256(p) for p in (SRC/'esdata').rglob('*.py')}
assert code_hashes, 'esdata source files not found'
signature_payload=json.dumps(dict(
    archive_hashes={c:sha256(path) for c,path in ARCHIVES.items()},
    code_hashes=code_hashes,channels=CHANNELS,
    profile='DefaultCSV+Simplify+SelectR+Cycles+CoolHeat3eps.2+CanonicalTempAlign200(175,390)+Subtraction1,-3,5'),sort_keys=True)
SIGNATURE=hashlib.sha256(signature_payload.encode()).hexdigest()
CACHE = OUT/'signals_cache.npz'
E5_CACHE = SRC/'outputs'/'E5_diagnostic_v2'/'signals_cache.npz'

def load_cache(path):
    if not path.is_file(): return None
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item())!=SIGNATURE: return None
        meta=pd.read_json(StringIO(str(z['metadata_json'].item())),orient='records')
        x=z['X'].astype(np.float32)
    meta['time']=pd.to_datetime(meta['time'])
    usable=meta[meta.split.ne('excluded')].sort_values(['card','time']).reset_index(drop=True)
    assert len(x)==len(usable) and x.shape[1:]==(12,200)
    return meta,usable,x

cache=load_cache(CACHE) or load_cache(E5_CACHE)
if cache is None:
    sys.path.insert(0,str(SRC))
    from esdata import EsmellDataset, Chain
    from esdata.transforms import (DefaultCSVPreprocessing,SimplifyLabels,SelectRChannels,
         UserLabelCycleSegments,CoolingHeatingSegments,CanonicalTempAlign,CanonicalSubtractionFeatures)
    rows,signals=[],{}
    for card,path in ARCHIVES.items():
        ds=EsmellDataset(root_path=DATA_DIR,subfolders=path.name,csv_repair='strict',archive_cache='disk',preload=True)
        pipe=ds >> Chain(DefaultCSVPreprocessing(),SimplifyLabels(),SelectRChannels(include=CHANNELS),
            UserLabelCycleSegments(),CoolingHeatingSegments(temp_col='T_mean',smooth_window=3,eps=.2),
            CanonicalTempAlign(n_grid=200,temp_range=(175.,390.),normalize=False,conductance=True),
            CanonicalSubtractionFeatures(features=((1,-3,5),((1,-3,5),(0,-2,4)))))
        for sid in ds.ids:
            m=ds.metadata(sid);t=pd.Timestamp(m['datetime'])
            split=('test' if card==146 else 'train' if t<pd.Timestamp('2026-09-28') else
                   'validation' if t<pd.Timestamp('2026-09-29') else 'excluded')
            label=str(m['classname']).lower()
            rows.append(dict(card=card,sample_id=str(sid),filename=m['filename'],time=t,
                             split=split,recipe=label))
            if split!='excluded':
                a=np.asarray(pipe.features(sid),dtype=np.float32)
                if a.shape!=(12,200) or not np.isfinite(a).all(): raise ValueError((sid,a.shape))
                signals[(card,str(sid))]=a
    meta=pd.DataFrame(rows)
    usable=meta[meta.split.ne('excluded')].sort_values(['card','time']).reset_index(drop=True)
    X=np.stack([signals[(r.card,r.sample_id)] for r in usable.itertuples()])
    np.savez_compressed(CACHE,signature=np.array(SIGNATURE),
        metadata_json=np.array(meta.to_json(orient='records',date_format='iso')),X=X)
else:
    meta,usable,X=cache
    print('Reused signature-verified cache.')

PAT=re.compile(r'^(acetone|isopropanol|pentane)\[([0-9]+(?:\.[0-9]+)?)ppb\]$')
def label_to_concentrations(label):
    if label=='air': return np.array([np.nan]*3,dtype=float)
    out=dict.fromkeys(GASES,np.nan)
    for part in label.lower().split('+'):
        m=PAT.fullmatch(part.strip())
        if m is None or np.isfinite(out.get(m.group(1),np.nan)): raise ValueError(f'Invalid label: {label}')
        out[m.group(1)]=float(m.group(2))
    return np.array([out[g] for g in GASES],dtype=float)

C=np.vstack([label_to_concentrations(r.recipe) for r in usable.itertuples()])
P=np.isfinite(C).astype(np.float32)        # TRUE presence, oracle input in E6 only
L=np.where(P>0,np.log10(np.where(P>0,C,1)),np.nan)
N=P.sum(1).astype(int)
TR=np.flatnonzero(usable.split.eq('train')); VA=np.flatnonzero(usable.split.eq('validation'))
TE=np.flatnonzero(usable.split.eq('test'))
assert (len(TR),len(VA),len(TE))==(2409,152,916)
assert X.shape==(3477,12,200) and np.isfinite(X).all()
assert (N[TR]>=2).sum()==154 and (N[VA]>=2).sum()==125 and (N[TE]>=2).sum()==98
assert (N[TE]==0).sum()==203
assert np.array_equal(P, (P.astype(bool)).astype(float))
assert not usable[['card','sample_id']].duplicated().any()
assert set(usable.split)=={'train','validation','test'}
assert np.all(C[P>0]>0)
usable.to_csv(OUT/'split_manifest.csv',index=False)

seen=np.zeros_like(P,dtype=bool)
for j in range(3):
    train_levels=np.unique(C[TR,j][P[TR,j]>0])
    seen[:,j]=(P[:,j]>0)&np.isin(C[:,j],train_levels)
for split,idx in [('train',TR),('validation',VA),('test',TE)]:
    print(split,len(idx),'pure',sum(N[idx]==1),'mixtures',sum(N[idx]>=2),
          'positives:',{g:int(P[idx,j].sum()) for j,g in enumerate(GASES)})
for j,g in enumerate(GASES):
    missing=sorted(set(C[TE,j][P[TE,j]>0])-set(C[TR,j][P[TR,j]>0]))
    print('Test-only levels',g,missing,'count:',int(((P[TE,j]>0)&~seen[TE,j]).sum()))
assert (int(((P[TE,1]>0)&~seen[TE,1]).sum()),int(((P[TE,2]>0)&~seen[TE,2]).sum()))==(40,40)

## 2. Regression target and metric

Fit only positive component targets. The principal score equally averages log-MAE over three gases and pure/mixed exposures.

In [ ]:
def stats192(a):
    z=[a.mean(2),a.std(2),a.min(2),a.max(2),np.median(a,axis=2),np.quantile(abs(a),.9,axis=2)]
    z.extend(t.mean(2) for t in np.array_split(a,10,axis=2))
    return np.concatenate(z,axis=1).astype(np.float32)
F=np.concatenate([stats192(X),P],axis=1)
assert F.shape==(len(X),195)
TRAIN_LO=np.array([np.nanmin(L[TR,j]) for j in range(3)])
TRAIN_HI=np.array([np.nanmax(L[TR,j]) for j in range(3)])
def bounded(p):return np.clip(p,TRAIN_LO,TRAIN_HI)
MU=X[TR].mean((0,2),keepdims=True); SD=np.maximum(X[TR].std((0,2),keepdims=True),1e-6)
XN=((X-MU)/SD).astype(np.float32)

def gas_indices(idx,j): return np.asarray(idx)[P[idx,j]>0]
def gas_weights(idx,j):
    pos=gas_indices(idx,j);is_pure=N[pos]==1
    a,b=int(is_pure.sum()),int((~is_pure).sum())
    if a==0 or b==0: raise ValueError(f'Gas {GASES[j]} lacks pure or mixed calibration')
    w=np.where(is_pure,.5/a,.5/b).astype(float)
    return pos,w/w.mean()

def balanced_log_mae(idx,pred):
    pred=bounded(pred)
    vals=[]
    for j in range(3):
        for kind in (1,2):  # 1 pure; 2 any mixture
            sub=np.asarray(idx)[(P[idx,j]>0)&((N[idx]==1) if kind==1 else (N[idx]>=2))]
            if len(sub)==0: return np.nan
            vals.append(np.abs(pred[sub,j]-L[sub,j]).mean())
    return float(np.mean(vals))

def seeded(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

# Optional source-independent numerical guards: no target from test in any scaler or weight.
for j in range(3):
    pos,w=gas_weights(TR,j)
    assert np.all(np.isfinite(L[pos,j])) and np.isclose(w[N[pos]==1].sum(),w[N[pos]>=2].sum())

## 3. Signal-free and classical baselines

Compare gas medians, composition medians, Ridge, RBF-SVR, and ExtraTrees using validation-only model selection.

In [ ]:
from functools import partial
from sklearn.base import clone

CONFIGS={
    'Ridge': [('alpha='+str(v),Ridge(alpha=v)) for v in (1.,30.,300.)],
    'SVR': [('C='+str(v),SVR(C=v,epsilon=.07,gamma='scale')) for v in (1.,10.)],
    'ExtraTrees': [('leaf='+str(v),ExtraTreesRegressor(n_estimators=300,min_samples_leaf=v,
                                  max_features='sqrt',random_state=42,n_jobs=-1)) for v in (2,8)]
}

def weighted_median(values,weights):
    order=np.argsort(values);v=np.asarray(values)[order];w=np.asarray(weights)[order]
    return float(v[np.searchsorted(np.cumsum(w),.5*w.sum())])

def make_median_predictions(conditioned):
    yhat=np.full_like(L,np.nan)
    for j in range(3):
        pos,w=gas_weights(TR,j)
        baseline=weighted_median(L[pos,j],w)
        groups={tuple(P[i].astype(int)):[] for i in TR if P[i,j]}
        for i in TR:
            if P[i,j]:groups[tuple(P[i].astype(int))].append(L[i,j])
        med={k:np.median(v) for k,v in groups.items()}
        for i in np.flatnonzero(P[:,j]>0):
            yhat[i,j]=med.get(tuple(P[i].astype(int)),baseline) if conditioned else baseline
    return yhat


def fit_classical(name,train_idx=TR,search=True):
    predictions=np.full_like(L,np.nan)
    decisions=[];fitted={}
    for j,g in enumerate(GASES):
        pos,w=gas_weights(train_idx,j)
        scaler=StandardScaler().fit(F[pos]);xtrain=scaler.transform(F[pos]);target=L[pos,j]
        # models are evaluated on all positive instances, never only on test.
        allpos=np.flatnonzero(P[:,j]>0);xall=scaler.transform(F[allpos])
        candidates=CONFIGS[name] if search else [CONFIGS[name][0]]
        best=None
        for key,base in candidates:
            model=clone(base);model.fit(xtrain,target,sample_weight=w)
            va=gas_indices(VA,j);yp=np.clip(model.predict(scaler.transform(F[va])),TRAIN_LO[j],TRAIN_HI[j])
            pure=(N[va]==1);mix=~pure
            loss=.5*(np.abs(yp[pure]-L[va[pure],j]).mean()+np.abs(yp[mix]-L[va[mix],j]).mean())
            if best is None or loss<best[0]-1e-12:best=(float(loss),key,model)
        _,choice,m=best
        predictions[allpos,j]=m.predict(xall)
        decisions.append(dict(model=name,gas=g,selection=choice,val_balanced_log_mae=best[0],n_train=len(pos)))
        fitted[g]={'scaler':scaler,'model':m}
    return predictions,decisions,fitted

## 4. Neural calibration

Compare a compact CNN and CNN with attention on the same signals. Report three seeds and train-only scaling.

In [ ]:
class CNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Conv1d(12,32,7,padding=3),nn.BatchNorm1d(32),nn.GELU(),nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.MaxPool1d(2),
            nn.Conv1d(64,96,3,padding=1),nn.BatchNorm1d(96),nn.GELU())
        self.head=nn.Sequential(nn.Linear(96*2+3,96),nn.GELU(),nn.Dropout(.2),nn.Linear(96,3))
    def forward(self,x,p):
        z=self.encoder(x)
        return self.head(torch.cat([z.mean(-1),z.amax(-1),p],dim=1))

class ConvAttentionRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Conv1d(12,32,7,padding=3),nn.BatchNorm1d(32),nn.GELU(),nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.MaxPool1d(2))
        self.pos=nn.Parameter(torch.zeros(1,50,64))
        layer=nn.TransformerEncoderLayer(d_model=64,nhead=4,dim_feedforward=128,
                       dropout=.1,activation='gelu',batch_first=True,norm_first=True)
        self.attention=nn.TransformerEncoder(layer,num_layers=2,enable_nested_tensor=False)
        self.head=nn.Sequential(nn.Linear(64+3,64),nn.GELU(),nn.Dropout(.2),nn.Linear(64,3))
    def forward(self,x,p):
        z=self.encoder(x).transpose(1,2)
        z=self.attention(z+self.pos).mean(1)
        return self.head(torch.cat([z,p],dim=1))

def fit_neural(name,seed):
    seeded(seed)
    net=(CNNRegressor() if name=='CNN' else ConvAttentionRegressor()).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=5e-4,weight_decay=1e-4)
    # one contribution per observed gas; each gas and each pure/mixed group has equal total weight
    w=np.zeros_like(P,dtype=np.float32)
    for j in range(3):
        pos,wj=gas_weights(TR,j);w[pos,j]=wj
    w=w/max(w[TR].sum()/len(TR),1e-6)
    y=np.nan_to_num(L,nan=0.).astype(np.float32)
    ds=TensorDataset(torch.from_numpy(XN[TR]),torch.from_numpy(P[TR]),
                     torch.from_numpy(y[TR]),torch.from_numpy(w[TR]))
    best=np.inf;best_state=None;best_epoch=None;stale=0;hist=[]
    def predict(index):
        net.eval();res=[]
        with torch.no_grad():
            for ids in np.array_split(np.asarray(index),max(1,int(np.ceil(len(index)/256)))):
                x=torch.tensor(XN[ids],device=DEVICE);p=torch.tensor(P[ids],device=DEVICE)
                res.append(net(x,p).cpu().numpy())
        a=np.concatenate(res).astype(float);a[P[index]==0]=np.nan
        return a
    for epoch in range(1,MAX_EPOCHS+1):
        net.train();losses=[]
        loader=DataLoader(ds,batch_size=BATCH,shuffle=True,num_workers=0,
                          generator=torch.Generator().manual_seed(seed+epoch))
        for x,p,t,weight in loader:
            x,p,t,weight=[v.to(DEVICE) for v in (x,p,t,weight)]
            opt.zero_grad()
            raw=nn.functional.smooth_l1_loss(net(x,p),t,reduction='none',beta=.15)
            loss=(raw*weight).sum()/weight.sum().clamp(min=1e-6)
            loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),5.0);opt.step()
            losses.append(float(loss.detach().cpu()))
        va_pred=predict(VA)
        pred_va=np.full_like(L,np.nan);pred_va[VA]=va_pred
        val=balanced_log_mae(VA,pred_va)
        if epoch==1 or epoch%5==0:
            pred_tr=np.full_like(L,np.nan);pred_tr[TR]=predict(TR)
            tr_loss=balanced_log_mae(TR,pred_tr)
        else:tr_loss=np.nan
        hist.append(dict(model=name,seed=seed,epoch=epoch,train_macro6_log_mae=tr_loss,
                         validation_macro6_log_mae=val,train_batch_huber=np.mean(losses)))
        if np.isfinite(val) and val<best-1e-4:
            best,best_epoch,best_state,stale=val,epoch,copy.deepcopy(net.state_dict()),0
        else:stale+=1
        if stale>=PATIENCE:break
    if best_state is None:raise RuntimeError(f'{name} failed to optimize')
    net.load_state_dict(best_state);out=predict(np.arange(len(X)))
    state={'state_dict':{k:v.cpu() for k,v in best_state.items()},'channel_mean':MU,'channel_std':SD,
           'name':name,'seed':seed,'best_epoch':best_epoch}
    torch.save(state,OUT/f'{name}_{seed}.pt')
    return out,hist,best_epoch,int(sum(v.numel() for v in net.parameters()))

for m in (CNNRegressor(),ConvAttentionRegressor()):
    assert m(torch.zeros(2,12,200),torch.ones(2,3)).shape==(2,3)
    print(type(m).__name__,sum(v.numel() for v in m.parameters()),'parameters')

## 5. Model fitting

Use the original training cohort and select checkpoints or regularization solely on validation. Also compare fixed ExtraTrees trained with and without mixtures.

In [ ]:
RUN_NEURAL=True                 # set False only for quick debugging, not a final article run
RUN_PURE_ONLY_CONTROL=True
R={};decisions=[];history=[];summary_info=[]
R['GasMedian']=make_median_predictions(False)
R['CompositionMedian']=make_median_predictions(True)
for name in ('Ridge','SVR','ExtraTrees'):
    t=time.perf_counter()
    pred,choice,fitted=fit_classical(name)
    R[name]=pred;decisions+=choice
    joblib.dump(fitted,OUT/f'{name}.joblib')
    summary_info.append(dict(model=name,seed=42,fit_seconds=time.perf_counter()-t))
    print(name,'validation macro6 log-MAE:',round(balanced_log_mae(VA,pred),4))
if RUN_PURE_ONLY_CONTROL:
    # Matched fixed-hyperparameter control: train on pure only vs all positive exposures.
    full,pure=np.full_like(L,np.nan),np.full_like(L,np.nan)
    for j,g in enumerate(GASES):
        allpos=np.flatnonzero(P[:,j]>0)
        for mode,dest in [('all',full),('pure',pure)]:
            if mode=='all': pos,w=gas_weights(TR,j)
            else: pos=TR[(P[TR,j]>0)&(N[TR]==1)];w=np.ones(len(pos))
            scaler=StandardScaler().fit(F[pos]);train=scaler.transform(F[pos])
            model=ExtraTreesRegressor(n_estimators=300,min_samples_leaf=2,
                                      max_features='sqrt',random_state=42,n_jobs=-1)
            model.fit(train,L[pos,j],sample_weight=w)
            dest[allpos,j]=model.predict(scaler.transform(F[allpos]))
    R['ExtraTrees_full_fixed']=full
    R['ExtraTrees_pure_only']=pure
if RUN_NEURAL:
    for name in ('CNN','ConvAttention'):
        for seed in SEEDS:
            t=time.perf_counter();pred,h,best,nparams=fit_neural(name,seed)
            R[f'{name}:{seed}']=pred
            history+=h
            summary_info.append(dict(model=name,seed=seed,best_epoch=best,
                                     parameters=nparams,fit_seconds=time.perf_counter()-t))
            print(name,seed,'best epoch',best,'validation macro6:',round(balanced_log_mae(VA,pred),4))

pd.DataFrame(decisions).to_csv(OUT/'hyperparameters.csv',index=False)
pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
pd.DataFrame(summary_info).to_csv(OUT/'model_costs.csv',index=False)

## 6. Held-out errors

Report log-MAE by gas and by pure/mixed condition, raw ppb error, clipping frequency, and performance on previously unseen nominal levels.

In [ ]:
LONG=[]
for run,pred in R.items():
    base,seed=(run.split(':')[0],int(run.split(':')[1])) if ':' in run else (run,42)
    for split,idx in [('train',TR),('validation',VA),('test',TE)]:
        for j,g in enumerate(GASES):
            for i in gas_indices(idx,j):
                z_raw=float(pred[i,j]);z=float(np.clip(z_raw,TRAIN_LO[j],TRAIN_HI[j]))
                actual=float(L[i,j]); actual_ppb=float(C[i,j])
                if not np.isfinite(z): raise ValueError(f'Nonfinite prediction: {run}/{g}/{i}')
                pred_ppb=float(np.power(10.,z))
                if not np.isfinite(pred_ppb):raise ValueError(f'Invalid ppb: {run}/{g}/{i}')
                LONG.append(dict(model=base,seed=seed,split=split,card=int(usable.iloc[i].card),
                    sample_id=usable.iloc[i].sample_id,day=str(usable.iloc[i].time.date()),
                    recipe=usable.iloc[i].recipe,gas=g,n_components=int(N[i]),
                    kind='pure' if N[i]==1 else 'mixture',level_seen_in_train=bool(seen[i,j]),
                    actual_ppb=actual_ppb,pred_ppb=pred_ppb,actual_log=actual,raw_pred_log=z_raw,
                    clipped=bool(abs(z_raw-z)>1e-10),pred_log=z,abs_log_error=abs(actual-z),abs_ppb_error=abs(actual_ppb-pred_ppb)))
LONG=pd.DataFrame(LONG)
assert len(LONG)>0 and not LONG.duplicated(['model','seed','split','sample_id','gas']).any()
LONG.to_csv(OUT/'predictions_long.csv',index=False)

def metric(p):
    if p.empty:return dict(n=0,log_mae=np.nan,log_rmse=np.nan,log_r2=np.nan,mae_ppb=np.nan)
    yy=p.actual_log.to_numpy();zz=p.pred_log.to_numpy()
    return dict(n=len(p),log_mae=float(np.mean(abs(yy-zz))),
                log_rmse=float(np.sqrt(np.mean((yy-zz)**2))),
                log_r2=float(r2_score(yy,zz)) if len(p)>1 and yy.std()>1e-12 else np.nan,
                mae_ppb=float(p.abs_ppb_error.mean()))

DETAIL=[];SUMMARY=[]
for (model,seed,split),df in LONG.groupby(['model','seed','split'],sort=False):
    m={g:{kind:metric(df[(df.gas==g)&(df.kind==kind)]) for kind in ('pure','mixture')} for g in GASES}
    cells=[m[g][k]['log_mae'] for g in GASES for k in ('pure','mixture')]
    assert all(np.isfinite(v) for v in cells)
    overall=metric(df)
    SUMMARY.append(dict(model=model,seed=seed,split=split,n_positive=len(df),
                macro6_log_mae=float(np.mean(cells)),log_mae=overall['log_mae'],
                mae_ppb=overall['mae_ppb'],clip_rate=float(df.clipped.mean()),
                pure_macro_log_mae=float(np.mean([m[g]['pure']['log_mae'] for g in GASES])),
                mixture_macro_log_mae=float(np.mean([m[g]['mixture']['log_mae'] for g in GASES]))))
    for g in GASES:
        for kind in ('pure','mixture'):
            DETAIL.append(dict(model=model,seed=seed,split=split,gas=g,kind=kind,**m[g][kind]))
SUMMARY=pd.DataFrame(SUMMARY);DETAIL=pd.DataFrame(DETAIL)
SUMMARY.to_csv(OUT/'scores.csv',index=False)
DETAIL.to_csv(OUT/'by_gas_pure_mix.csv',index=False)

# New nominal levels exist in *pure* test only; never collapse with mixture claims.
LEVEL=LONG[(LONG.split=='test')&(LONG.kind=='pure')].groupby(
    ['model','seed','gas','level_seen_in_train']).apply(metric,include_groups=False)
LEVEL=pd.json_normalize(LEVEL.to_list()).set_axis(LEVEL.index).reset_index()
LEVEL.to_csv(OUT/'seen_vs_unseen_levels.csv',index=False)
AB=DETAIL[(DETAIL.split=='test')&(DETAIL.kind=='mixture')&DETAIL.model.isin(['ExtraTrees_full_fixed','ExtraTrees_pure_only'])].copy()
AB.to_csv(OUT/'pure_to_mixture_control.csv',index=False)
DAY=[]
for keys,d in LONG[LONG.split=='test'].groupby(['model','seed','day','gas','kind']):
    DAY.append(dict(**dict(zip(['model','seed','day','gas','kind'],keys)),**metric(d)))
pd.DataFrame(DAY).to_csv(OUT/'by_day.csv',index=False)

print('\nHeld-out campaign 146:')
print(SUMMARY[SUMMARY.split=='test'][['model','seed','macro6_log_mae','pure_macro_log_mae',
                    'mixture_macro_log_mae','mae_ppb','clip_rate']].sort_values('macro6_log_mae').to_string(index=False,float_format=lambda x:f'{x:.3f}'))

## 7. Paper tables and figures

Write the fixed model panel and three-gas parity plots. Seed variation does not represent independent-day uncertainty.

In [ ]:
# Deterministic publication table, with no test-set-dependent hyperparameter or figure selection.
TEST=SUMMARY[SUMMARY.split=='test'].copy()
model_order=['GasMedian','CompositionMedian','Ridge','SVR','ExtraTrees','CNN','ConvAttention']
def cell_fmt(df,column):
    a=df[column].to_numpy(dtype=float)
    return f'{a.mean():.3f}' if len(a)==1 else f'{a.mean():.3f} $\\pm$ {a.std(ddof=1):.3f}'
rows=[]
for name in model_order:
    sub=TEST[TEST.model==name]
    if sub.empty:continue
    rows.append([name,cell_fmt(sub,'macro6_log_mae'),cell_fmt(sub,'pure_macro_log_mae'),
                 cell_fmt(sub,'mixture_macro_log_mae'),cell_fmt(sub,'mae_ppb')])
def esc(s):return str(s).replace('_',r'\_')
lines=[r'\begin{table}[t]',
    r'\caption{E6 oracle-composition concentration regression on held-out campaign 146. Decimal-log MAE is lower-is-better; six gas-by-type cells are macro-averaged. Neural results are mean $\pm$ SD across seeds, not across days.}',
    r'\label{tab:e6}',r'\centering\small',r'\begin{tabular}{@{}lrrrr@{}}',r'\toprule',
    r'Model & Macro-6 & Pure & Mixtures & MAE (ppb) \\',r'\midrule']
lines += [esc(a)+' & '+' & '.join(b)+' \\\\' for a,*b in rows]
lines += [r'\bottomrule',r'\end{tabular}',r'\end{table}']
(OUT/'E6_table.tex').write_text('\n'.join(lines)+'\n',encoding='utf-8')

fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for j,g in enumerate(GASES):
    ax=axes[j]
    for model,marker in [('Ridge','o'),('ExtraTrees','x')]:
        d=LONG[(LONG.split=='test')&(LONG.model==model)&(LONG.gas==g)]
        ax.scatter(d.actual_log,d.pred_log,s=10,alpha=.35,label=model,marker=marker)
    lims=[min(LONG[(LONG.split=='test')&(LONG.gas==g)].actual_log.min(),
              LONG[(LONG.split=='test')&(LONG.gas==g)].pred_log.min()),
          max(LONG[(LONG.split=='test')&(LONG.gas==g)].actual_log.max(),
              LONG[(LONG.split=='test')&(LONG.gas==g)].pred_log.max())]
    ax.plot(lims,lims,'k--',lw=1);ax.set_title(g)
    ax.set_xlabel('Nominal log10(ppb)');ax.set_ylabel('Predicted log10(ppb)')
axes[0].legend(frameon=False,fontsize=8)
fig.tight_layout();fig.savefig(OUT/'E6_parity.pdf');plt.close(fig)
if history:
    h=pd.DataFrame(history);fig,axes=plt.subplots(1,2,figsize=(9,3.4))
    for ax,name in zip(axes,['CNN','ConvAttention']):
        sub=h[h.model==name]
        for seed,part in sub.groupby('seed'):
            ax.plot(part.epoch,part.validation_macro6_log_mae,label=f'val seed {seed}',alpha=.7)
            ax.plot(part.epoch,part.train_macro6_log_mae,linestyle=':',alpha=.5)
        ax.set_title(name);ax.set_xlabel('Epoch');ax.set_ylabel('Macro-6 log-MAE');ax.legend(fontsize=7,frameon=False)
    fig.tight_layout();fig.savefig(OUT/'E6_learning_curves.pdf');plt.close(fig)

text=['# E6 · Regression report (oracle composition)','',
      'Test: campaign 146; train/validation: campaign 130. Values are *nominal* concentrations.',
      'No test concentration is supplied as an input. True presence IS supplied as oracle input.',
      'Primary metric: equal-weight mean of six gas × {pure,mixture} log10-MAEs.',
      'The 146 campaign has only one day of mixture exposures; do not compute mixture-day confidence intervals.','',
      '## Held-out scores','',TEST[['model','seed','macro6_log_mae','pure_macro_log_mae','mixture_macro_log_mae','mae_ppb']].to_markdown(index=False),
      '', '## New concentration levels: separate interpolation test','',
      LEVEL[(LEVEL.level_seen_in_train==False)].to_markdown(index=False),
      '', '## Validation vs test','',
      SUMMARY[SUMMARY.split.isin(['validation','test'])][['model','seed','split','macro6_log_mae']].to_markdown(index=False),
      '', '## Interpretation safeguards',
      '- Compare against both signal-free medians, not just other regressors.',
      '- Report pure and mixture errors separately, and explain that the composition input is oracle.',
      '- Seen/novel levels are within the calibration range; this is interpolation, not extrapolation.',
      '- The fixed-leaf ExtraTrees pair is a diagnostic, not test-selected. It changes the available calibration population and cannot isolate a physical causal mechanism.',
      '- A low regression error here does NOT imply that E7 can infer the true mixture composition.']
(OUT/'E6_report.md').write_text('\n'.join(text)+'\n',encoding='utf-8')
config={'task':'E6 oracle concentration regression','gases':GASES,'seed':SEEDS,
        'splits':dict(train=len(TR),validation=len(VA),test=len(TE)),
        'cache_sha256':SIGNATURE,'pipeline':'CanonicalSubtractionFeatures (E5 exact)',
        'model_grid':{k:[name for name,_ in v] for k,v in CONFIGS.items()},
        'train_only_scaling':True,'model_selection':'validation six-cell log-MAE',
        'concentration_targets':'nominal positive log10(ppb)','oracle_presence_input':True,
        'no_refit_after_validation':True,'test_previously_inspected_in_E5':True,
        'run_neural':RUN_NEURAL,'pure_only_diagnostic':RUN_PURE_ONLY_CONTROL,
        'prediction_calibration_bounds_log10':{g:[float(TRAIN_LO[i]),float(TRAIN_HI[i])] for i,g in enumerate(GASES)}}
(OUT/'run_config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('Wrote:',OUT/'E6_table.tex',OUT/'E6_parity.pdf',OUT/'E6_report.md')